# D+7 활동 예측용 질문 카테고리 피처 생성

## 분석 목적

질문 텍스트를 SBERT 임베딩과 K-Means로 의미 기반 군집화한 뒤, 공통 Base Mart 사용자의 **최초 PING 발신 후 24시간 질문 행동**을 사용자 단위 피처로 집계한다.

- 최종 타깃: `retained_d7` — T0+7일 이상 T0+8일 미만 PING 발신 여부
- 분석 단위: 사용자 1명당 1행
- 피처 관찰 구간: `[feature_start_at, feature_end_at)` = `[T0, T0+24시간)`
- 결과 테이블: `your-gcp-project.mart.fct_ml_category_24h`

이 노트북은 질문 추천 모델이나 D+7 예측 모델을 학습하지 않는다. SBERT와 K-Means는 질문 텍스트에 카테고리를 붙여 **피처를 생성하기 위한 전처리 도구**로만 사용한다.

> 카테고리는 반드시 `question_piece_id → question_id → question_text → cluster_id → category_name` 경로로 연결한다. `question_piece_id % 10` 같은 임의 분류 방식은 사용하지 않는다.

## 1. 환경 설정

Colab에서 Google 계정을 인증하고 BigQuery 클라이언트를 생성한다.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "your-gcp-project"
client = bigquery.Client(project=PROJECT_ID)

pd.set_option("display.max_columns", None)
print("BigQuery 연결 완료")

## 2. 질문 연결 경로 검증

PING 발신 기록에서 질문 텍스트까지 다음 순서로 연결한다.

```text
raw_userquestionrecord.question_piece_id
→ polls_questionpiece.id
→ polls_questionpiece.question_id
→ stg_question.question_id
→ question_text_norm
```

아래 QA에서 전체 기록 수와 고유 기록 수가 같고 네 가지 오류 건수가 모두 0이어야 한다. 이 검증은 조인 중복으로 PING 수가 부풀거나 질문이 잘못 연결되는 문제를 막기 위한 단계다.

In [ ]:
query = """
SELECT
    COUNT(*) AS record_count,
    COUNT(DISTINCT r.id) AS unique_record_count,

    -- question_piece 연결 실패
    COUNTIF(
        r.question_piece_id IS NOT NULL
        AND qp.id IS NULL
    ) AS unmatched_question_piece_count,

    -- question 연결 실패
    COUNTIF(
        qp.question_id IS NOT NULL
        AND q.question_id IS NULL
    ) AS unmatched_question_count,

    -- 발신 기록과 question_piece의 question_id 불일치
    COUNTIF(
        r.question_id IS NOT NULL
        AND qp.question_id IS NOT NULL
        AND r.question_id != qp.question_id
    ) AS question_id_mismatch_count,

    -- 질문 텍스트 결측
    COUNTIF(
        q.question_id IS NOT NULL
        AND NULLIF(TRIM(q.question_text), '') IS NULL
    ) AS missing_question_text_count

FROM `your-gcp-project.raw.raw_userquestionrecord` r

LEFT JOIN `your-gcp-project.raw.polls_questionpiece` qp
    ON r.question_piece_id = qp.id

LEFT JOIN `your-gcp-project.stage.stg_question` q
    ON qp.question_id = q.question_id
"""

question_mapping_qa = client.query(query).to_dataframe()
question_mapping_qa.T

## 3. 유효 질문 추출 및 중복 텍스트 처리

Stage에서 유효하며 정규화 텍스트가 존재하는 질문만 가져온다. 동일한 문구가 여러 `question_id`에 반복될 수 있으므로 SBERT에는 고유한 `question_text_norm`만 입력한다.

질문 ID를 삭제하는 것은 아니다. 고유 텍스트를 한 번만 군집화한 뒤 같은 문구를 가진 모든 질문 ID에 동일한 군집을 다시 연결한다.

In [ ]:
query = """
SELECT
    question_id,
    question_text,
    question_text_norm,
    is_valid_question,
    created_at
FROM `your-gcp-project.stage.stg_question`
WHERE is_valid_question = TRUE
  AND NULLIF(TRIM(question_text_norm), '') IS NOT NULL
ORDER BY question_id
"""

question_master_df = client.query(query).to_dataframe()

print("유효 질문 수:", len(question_master_df))
print("고유 question_id:", question_master_df["question_id"].nunique())
print(
    "고유 질문 텍스트:",
    question_master_df["question_text_norm"].nunique()
)

question_master_df.head(20)

In [ ]:
question_master_qa = pd.DataFrame({
    "row_count": [len(question_master_df)],
    "unique_question_id": [
        question_master_df["question_id"].nunique()
    ],
    "unique_question_text": [
        question_master_df["question_text_norm"].nunique()
    ],
    "duplicate_question_id": [
        question_master_df["question_id"].duplicated().sum()
    ],
    "duplicate_question_text": [
        question_master_df["question_text_norm"].duplicated().sum()
    ]
})

question_master_qa.T

In [ ]:
question_text_df = (
    question_master_df[
        ["question_text_norm"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

question_text_df["text_id"] = question_text_df.index

print("SBERT 임베딩 대상:", len(question_text_df))
question_text_df.head()

## 4. SBERT 임베딩

한국어 문장을 의미 벡터로 변환한다. 정규화 임베딩을 사용하며, 질문 텍스트 외에 사용자 행동이나 D+7 타깃은 입력하지 않는다.

In [ ]:
!pip install -q sentence-transformers

import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
print("임베딩 대상 질문:", len(question_text_df))

MODEL_NAME = "jhgan/ko-sroberta-multitask"

sbert_model = SentenceTransformer(MODEL_NAME)

question_embeddings = sbert_model.encode(
    question_text_df["question_text_norm"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("임베딩 형태:", question_embeddings.shape)

## 5. 군집 수 검토 및 최종 군집 생성

`k=5~15`의 Silhouette Score를 비교한다. 점수는 군집 수 선택의 참고 지표이며, 최종적으로는 대표 질문을 사람이 읽었을 때 설명 가능한지도 함께 고려한다.

현재 데이터에서는 `k=5`의 점수가 가장 높았고, 최종 피처도 5개 군집 기준으로 고정한다.

In [ ]:
cluster_scores = []

for k in range(5, 16):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(question_embeddings)

    score = silhouette_score(
        question_embeddings,
        labels,
        metric="cosine"
    )

    cluster_scores.append({
        "k": k,
        "silhouette_score": score
    })

cluster_score_df = pd.DataFrame(cluster_scores)
cluster_score_df

In [ ]:
N_CLUSTERS = 5
RANDOM_STATE = 42

final_kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=10
)

cluster_labels = final_kmeans.fit_predict(
    question_embeddings
)

question_text_cluster_df = question_text_df.copy()
question_text_cluster_df["cluster_id"] = cluster_labels

print("군집화된 고유 질문 문구:", len(question_text_cluster_df))
print(
    "생성된 군집 수:",
    question_text_cluster_df["cluster_id"].nunique()
)

### 군집 크기와 대표 질문 확인

군집별 질문 수를 확인하고 각 중심에 가까운 대표 질문 15개를 출력한다. 군집 번호는 의미가 없는 식별자이므로 대표 질문을 검토한 뒤 카테고리명을 부여한다.

In [ ]:
cluster_size_df = (
    question_text_cluster_df
    .groupby("cluster_id")
    .size()
    .reset_index(name="question_text_count")
    .sort_values("cluster_id")
)

cluster_size_df

In [ ]:
cluster_distances = final_kmeans.transform(
    question_embeddings
)

representative_rows = []

for cluster_id in sorted(
    question_text_cluster_df["cluster_id"].unique()
):
    cluster_indexes = np.where(
        cluster_labels == cluster_id
    )[0]

    nearest_indexes = cluster_indexes[
        np.argsort(
            cluster_distances[
                cluster_indexes,
                cluster_id
            ]
        )[:15]
    ]

    for rank, idx in enumerate(nearest_indexes, start=1):
        representative_rows.append({
            "cluster_id": cluster_id,
            "rank": rank,
            "question_text_norm":
                question_text_cluster_df.iloc[idx][
                    "question_text_norm"
                ]
        })

cluster_review_df = pd.DataFrame(representative_rows)

In [ ]:
for cluster_id in sorted(
    cluster_review_df["cluster_id"].unique()
):
    print(f"\n===== Cluster {cluster_id} =====")

    display(
        cluster_review_df[
            cluster_review_df["cluster_id"] == cluster_id
        ][["rank", "question_text_norm"]]
    )

## 6. 카테고리 이름 부여 및 전체 질문 ID 연결

대표 질문을 검토하여 5개 군집에 해석 가능한 이름을 붙인다. 이 이름은 서비스의 공식 카테고리가 아니라 질문 텍스트에서 생성한 **잠재 의미 군집**이다.

군집 결과가 달라진 상태에서 이 셀만 재사용하면 잘못된 이름이 붙을 수 있으므로, SBERT 모델·군집 수·랜덤 시드를 변경했다면 대표 질문을 다시 검토해야 한다.

In [ ]:
category_name_map = {
    0: "긍정적 이미지·호감",
    1: "연애·데이트",
    2: "감정·내면·관계 심리",
    3: "매력·성격 특성",
    4: "친구·친밀감"
}

question_text_cluster_df["category_name"] = (
    question_text_cluster_df["cluster_id"]
    .map(category_name_map)
)

In [ ]:
question_cluster_map_df = (
    question_master_df[
        ["question_id", "question_text", "question_text_norm"]
    ]
    .merge(
        question_text_cluster_df[
            [
                "question_text_norm",
                "cluster_id",
                "category_name"
            ]
        ],
        on="question_text_norm",
        how="left",
        validate="many_to_one"
    )
)

print("전체 질문:", len(question_cluster_map_df))
print(
    "고유 question_id:",
    question_cluster_map_df["question_id"].nunique()
)
print(
    "카테고리 결측:",
    question_cluster_map_df["category_name"].isna().sum()
)

In [ ]:
mapping_qa = pd.DataFrame({
    "row_count": [len(question_cluster_map_df)],
    "unique_question_id": [
        question_cluster_map_df["question_id"].nunique()
    ],
    "cluster_null_count": [
        question_cluster_map_df["cluster_id"].isna().sum()
    ],
    "category_null_count": [
        question_cluster_map_df["category_name"].isna().sum()
    ]
})

mapping_qa.T

## 7. 질문–카테고리 매핑 테이블 저장

전체 질문 ID 4,969개에 `cluster_id`와 `category_name`을 연결하여 BigQuery에 저장한다.

결과 테이블: `your-gcp-project.mart.dim_ml_question_category`

In [ ]:
question_category_upload_df = (
    question_cluster_map_df[
        [
            "question_id",
            "question_text",
            "question_text_norm",
            "cluster_id",
            "category_name"
        ]
    ]
    .copy()
)

question_category_upload_df["question_id"] = (
    question_category_upload_df["question_id"]
    .astype("int64")
)

question_category_upload_df["cluster_id"] = (
    question_category_upload_df["cluster_id"]
    .astype("int64")
)

In [ ]:
table_id = (
    "your-gcp-project."
    "mart.dim_ml_question_category"
)

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

load_job = client.load_table_from_dataframe(
    question_category_upload_df,
    table_id,
    job_config=job_config
)

load_job.result()

print("질문 카테고리 매핑 테이블 저장 완료")

In [ ]:
query = """
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT question_id) AS question_count,
    COUNTIF(cluster_id IS NULL) AS cluster_null_count,
    COUNTIF(category_name IS NULL) AS category_null_count
FROM `your-gcp-project.mart.dim_ml_question_category`
"""

client.query(query).to_dataframe().T

## 8. 사용자 단위 첫 24시간 카테고리 피처 생성

공통 Base Mart의 `user_id`와 시간창을 기준으로 질문 발신 기록을 집계한다.

- 기준 시점: `first_ping_at`
- 포함 조건: `created_at >= feature_start_at AND created_at < feature_end_at`
- Base Mart를 최종 왼쪽 테이블로 사용하여 4,844명을 유지
- D+7 타깃 구간이나 전체 기간 누적값은 사용하지 않음

`category_diversity_24h`는 0~1 범위의 정규화 엔트로피다. 값이 클수록 여러 카테고리에 고르게 반응했음을 의미한다. `max_category_share_24h`는 가장 많이 사용한 카테고리의 비중이며 값이 클수록 특정 카테고리에 집중되었음을 의미한다.

In [ ]:
query = """
CREATE OR REPLACE TABLE
    `your-gcp-project.mart.fct_ml_category_24h` AS

WITH question_events_24h AS (
    SELECT
        b.user_id,
        r.id AS user_question_record_id,
        m.cluster_id,
        m.category_name

    FROM `your-gcp-project.mart.mart_ml_d7_base` b

    INNER JOIN
        `your-gcp-project.raw.raw_userquestionrecord` r
        ON b.user_id = r.user_id
       AND r.created_at >= b.feature_start_at
       AND r.created_at < b.feature_end_at

    LEFT JOIN
        `your-gcp-project.raw.polls_questionpiece` qp
        ON r.question_piece_id = qp.id

    LEFT JOIN
        `your-gcp-project.mart.dim_ml_question_category` m
        ON qp.question_id = m.question_id
),

-- 사용자별 전체 발신 및 매핑 수
user_totals AS (
    SELECT
        user_id,

        COUNT(user_question_record_id)
            AS total_question_ping_count_24h,

        COUNTIF(cluster_id IS NOT NULL)
            AS categorized_ping_count_24h,

        COUNTIF(cluster_id IS NULL)
            AS uncategorized_ping_count_24h

    FROM question_events_24h
    GROUP BY user_id
),

-- 사용자–카테고리별 발신 횟수
category_counts AS (
    SELECT
        user_id,
        cluster_id,
        category_name,
        COUNT(*) AS category_ping_count

    FROM question_events_24h

    WHERE cluster_id IS NOT NULL

    GROUP BY
        user_id,
        cluster_id,
        category_name
),

-- 카테고리 수, 집중도, 엔트로피 계산
category_stats AS (
    SELECT
        c.user_id,

        COUNT(*) AS category_count_24h,

        MAX(c.category_ping_count)
            AS max_category_ping_count,

        -SUM(
            SAFE_DIVIDE(
                c.category_ping_count,
                t.categorized_ping_count_24h
            )
            * LN(
                SAFE_DIVIDE(
                    c.category_ping_count,
                    t.categorized_ping_count_24h
                )
            )
        ) AS category_entropy

    FROM category_counts c

    INNER JOIN user_totals t
        USING (user_id)

    GROUP BY
        c.user_id,
        t.categorized_ping_count_24h
),

-- 대표 카테고리
dominant_category AS (
    SELECT
        user_id,
        cluster_id AS dominant_cluster_id_24h,
        category_name AS dominant_category_24h

    FROM category_counts

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY user_id
        ORDER BY
            category_ping_count DESC,
            cluster_id ASC
    ) = 1
),

-- 카테고리별 PING 발신 횟수
category_specific_counts AS (
    SELECT
        user_id,

        COUNTIF(cluster_id = 0)
            AS category_0_ping_count_24h,

        COUNTIF(cluster_id = 1)
            AS category_1_ping_count_24h,

        COUNTIF(cluster_id = 2)
            AS category_2_ping_count_24h,

        COUNTIF(cluster_id = 3)
            AS category_3_ping_count_24h,

        COUNTIF(cluster_id = 4)
            AS category_4_ping_count_24h

    FROM question_events_24h

    WHERE cluster_id IS NOT NULL

    GROUP BY user_id
)

SELECT
    b.user_id,

    COALESCE(
        t.total_question_ping_count_24h,
        0
    ) AS total_question_ping_count_24h,

    COALESCE(
        t.categorized_ping_count_24h,
        0
    ) AS categorized_ping_count_24h,

    COALESCE(
        t.uncategorized_ping_count_24h,
        0
    ) AS uncategorized_ping_count_24h,

    SAFE_DIVIDE(
        COALESCE(t.categorized_ping_count_24h, 0),
        COALESCE(t.total_question_ping_count_24h, 0)
    ) AS category_mapping_rate_24h,

    COALESCE(
        s.category_count_24h,
        0
    ) AS category_count_24h,

    COALESCE(
        d.dominant_cluster_id_24h,
        -1
    ) AS dominant_cluster_id_24h,

    COALESCE(
        d.dominant_category_24h,
        'NO_ACTIVITY'
    ) AS dominant_category_24h,

    -- 가장 많이 사용한 카테고리의 비율
    SAFE_DIVIDE(
        s.max_category_ping_count,
        t.categorized_ping_count_24h
    ) AS max_category_share_24h,

    -- 0~1 범위의 정규화 엔트로피
    CASE
    WHEN COALESCE(s.category_count_24h, 0) <= 1
        THEN 0.0

    ELSE LEAST(
        1.0,
        GREATEST(
            0.0,
            SAFE_DIVIDE(
                s.category_entropy,
                LN(s.category_count_24h)
            )
        )
    )
    END AS category_diversity_24h,

    COALESCE(
        c.category_0_ping_count_24h, 0
    ) AS category_0_ping_count_24h,

    COALESCE(
        c.category_1_ping_count_24h, 0
    ) AS category_1_ping_count_24h,

    COALESCE(
        c.category_2_ping_count_24h, 0
    ) AS category_2_ping_count_24h,

    COALESCE(
        c.category_3_ping_count_24h, 0
    ) AS category_3_ping_count_24h,

    COALESCE(
        c.category_4_ping_count_24h, 0
    ) AS category_4_ping_count_24h

FROM `your-gcp-project.mart.mart_ml_d7_base` b

LEFT JOIN user_totals t
    USING (user_id)

LEFT JOIN category_stats s
    USING (user_id)

LEFT JOIN dominant_category d
    USING (user_id)

LEFT JOIN category_specific_counts c
    USING (user_id)
"""

client.query(query).result()

print("질문 카테고리 피처 테이블 생성 완료")

## 9. 최종 QA

다음 조건을 모두 만족해야 한다.

1. `row_count = user_count = 4,844`
2. `duplicate_count = 0`
3. 전체 PING 수 = 카테고리 연결 PING 수 + 미연결 PING 수
4. 다섯 카테고리별 PING 수 합 = 카테고리 연결 PING 수
5. 다양성과 집중도 범위 오류 = 0

In [ ]:
query = """
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT user_id) AS user_count,
    COUNT(*) - COUNT(DISTINCT user_id)
        AS duplicate_count,

    COUNTIF(
        categorized_ping_count_24h
        + uncategorized_ping_count_24h
        != total_question_ping_count_24h
    ) AS total_count_error,

    COUNTIF(
        category_0_ping_count_24h
        + category_1_ping_count_24h
        + category_2_ping_count_24h
        + category_3_ping_count_24h
        + category_4_ping_count_24h
        != categorized_ping_count_24h
    ) AS category_sum_error,

    COUNTIF(
        category_diversity_24h < 0
        OR category_diversity_24h > 1
    ) AS diversity_range_error,

    COUNTIF(
        max_category_share_24h < 0
        OR max_category_share_24h > 1
    ) AS concentration_range_error

FROM `your-gcp-project.mart.fct_ml_category_24h`
"""

category_feature_qa = client.query(query).to_dataframe()
category_feature_qa.T

In [ ]:
query = """
SELECT *
FROM `your-gcp-project.mart.fct_ml_category_24h`
ORDER BY user_id
LIMIT 10
"""

category_feature_sample = client.query(query).to_dataframe()
category_feature_sample

## 10. 최종 피처 설명

| 피처 | 의미 | 비고 |
|---|---|---|
| `total_question_ping_count_24h` | 첫 24시간 전체 질문 PING 수 | PING 피처와 중복 여부 확인 |
| `categorized_ping_count_24h` | 카테고리가 연결된 PING 수 | QA·보조 피처 |
| `uncategorized_ping_count_24h` | 카테고리가 연결되지 않은 PING 수 | QA·보조 피처 |
| `category_mapping_rate_24h` | 전체 PING 중 카테고리 연결 비율 | QA·보조 피처 |
| `category_count_24h` | 사용한 고유 질문 카테고리 수 | 모델 후보 |
| `dominant_cluster_id_24h` | 가장 많이 사용한 군집 번호 | `dominant_category_24h`와 중복 사용 주의 |
| `dominant_category_24h` | 가장 많이 사용한 대표 카테고리 | 범주형 전처리 필요 |
| `max_category_share_24h` | 대표 카테고리의 발신 비중 | 집중도, 0~1 |
| `category_diversity_24h` | 카테고리 사용의 고른 정도 | 정규화 엔트로피, 0~1 |
| `category_0~4_ping_count_24h` | 카테고리별 PING 발신 횟수 | 모델 후보 |

### 모델링 시 주의사항

- `total_question_ping_count_24h`는 PING 담당의 전체 발신 횟수와 중복될 수 있으므로 최종 Mart에서 중복 여부를 확인한다.
- `dominant_cluster_id_24h`와 `dominant_category_24h`는 같은 정보를 표현하므로 둘을 동시에 숫자형 피처로 넣지 않는다. 대표 카테고리는 One-Hot Encoding 대상으로 사용한다.
- 카테고리 군집의 Silhouette Score가 높지 않으므로 공식 분류처럼 해석하지 않고 Ablation Test로 실제 기여도를 검증한다.
- 최종 모델 학습과 Train/Test 분할은 모든 피처 테이블을 합친 `mart_ml_d7_training`에서 공통으로 수행한다.